# 🚀 THÀNH VIÊN 3: FINE-TUNING PhoBERT TRÊN GOOGLE COLAB GPU T4
## Đề tài: Phát hiện đánh giá độc hại (Toxic Detection) trên sàn TMĐT Điện Máy Xanh

---
### 📌 HƯỚNG DẪN 3 BƯỚC THỰC HIỆN ĐƠN GIẢN (CHỈ MẤT 4 - 5 PHÚT):
1. **Bật GPU T4:** Vào menu **Runtime** (Thời gian chạy) -> **Change runtime type** (Thay đổi loại thời gian chạy) -> Chọn **T4 GPU** -> Bấm **Save**.
2. **Tải file dữ liệu lên:** Bấm vào biểu tượng **Thư mục (Files)** ở thanh công cụ bên trái -> Kéo thả file `colab_phobert_data.zip` vào đó.
3. **Chạy toàn bộ:** Vào menu **Runtime** -> Chọn **Run all** (hoặc bấm tổ hợp phím `Ctrl + F9`).
   - Chờ hệ thống huấn luyện khoảng **3 đến 4 phút**.
   - Sau khi chạy xong, trình duyệt sẽ **tự động tải về máy 2 file**: `phobert.csv` và `phobert_info.json`!

In [ ]:
# BƯỚC 1: KIỂM TRA GPU T4 VÀ CÀI ĐẶT THƯ VIỆN
!nvidia-smi
!pip install -q transformers accelerate
import torch
print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Device:', torch.cuda.get_device_name(0))
else:
    print('⚠️ CẢNH BÁO: Chưa bật GPU! Hãy vào Runtime -> Change runtime type -> T4 GPU!')

In [ ]:
# BƯỚC 2: GIẢI NÉN DỮ LIỆU ĐIỆN MÁY XANH CHUẨN (7.000 Train / 1.000 Val / 2.000 Test)
import zipfile, os
import pandas as pd

if os.path.exists('colab_phobert_data.zip'):
    with zipfile.ZipFile('colab_phobert_data.zip', 'r') as zf:
        zf.extractall('.')
    print('✅ Đã giải nén thành công train.csv, val.csv, test.csv!')
else:
    print('❌ LỖI: Chưa tìm thấy colab_phobert_data.zip. Bạn hãy kéo file này thả vào tab Files bên trái nhé!')

df_train = pd.read_csv('train.csv')
df_val   = pd.read_csv('val.csv')
df_test  = pd.read_csv('test.csv')

print(f'Train: {len(df_train):,} mẫu | Toxic: {df_train["label"].sum()} ({df_train["label"].mean()*100:.2f}%)')
print(f'Val  : {len(df_val):,} mẫu | Toxic: {df_val["label"].sum()} ({df_val["label"].mean()*100:.2f}%)')
print(f'Test : {len(df_test):,} mẫu | Toxic: {df_test["label"].sum()} ({df_test["label"].mean()*100:.2f}%)')

In [ ]:
# BƯỚC 3: KHỞI TẠO TOKENIZER VÀ PYTORCH DATASET
from transformers import AutoTokenizer
from torch.utils.data import Dataset, DataLoader

MODEL_NAME = 'vinai/phobert-base-v2'
print(f'Đang tải tokenizer {MODEL_NAME}...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class DMXToxicDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=64):
        self.texts = [str(t) if pd.notnull(t) else '' for t in texts]
        self.labels = list(labels)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        enc = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt'
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

train_ds = DMXToxicDataset(df_train['noi_dung_tien_xu_ly'], df_train['label'], tokenizer)
val_ds   = DMXToxicDataset(df_val['noi_dung_tien_xu_ly'], df_val['label'], tokenizer)
test_ds  = DMXToxicDataset(df_test['noi_dung_tien_xu_ly'], df_test['label'], tokenizer)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=64, shuffle=False)
test_loader  = DataLoader(test_ds, batch_size=64, shuffle=False)
print('✅ Khởi tạo DataLoader thành công!')

In [ ]:
# BƯỚC 4: THIẾT LẬP MÔ HÌNH VÀ CHIẾN LƯỢC COST-SENSITIVE TRAINING
import torch.nn as nn
from transformers import AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from torch.optim import AdamW

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model.to(device)

# Phạt nặng mẫu Toxic gấp 8 lần Non-toxic để cân bằng độ lệch tự nhiên 3.4%
class_weights = torch.tensor([1.0, 8.0], dtype=torch.float).to(device)
loss_fn = nn.CrossEntropyLoss(weight=class_weights)

EPOCHS = 3
optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(total_steps * 0.1), num_training_steps=total_steps)
print(f'✅ Đã sẵn sàng huấn luyện {EPOCHS} epochs trên {device}!')

In [ ]:
# BƯỚC 5: TIẾN HÀNH FINE-TUNING TOÀN PHẦN (MẤT ~3-4 PHÚT TRÊN T4)
import time, numpy as np
from sklearn.metrics import f1_score, precision_score, recall_score

best_val_f1 = -1.0
best_ckpt = 'best_phobert_model.pt'

print('=' * 75)
print('🚀 BẮT ĐẦU FINE-TUNING PhoBERT TRÊN GPU T4...')
print('=' * 75)

for epoch in range(EPOCHS):
    t0 = time.time()
    model.train()
    total_loss = 0.0
    for step, batch in enumerate(train_loader):
        input_ids = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        optimizer.zero_grad()
        out = model(input_ids=input_ids, attention_mask=mask)
        loss = loss_fn(out.logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    # Đánh giá trên Validation
    model.eval()
    v_preds, v_trues = [], []
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            out = model(input_ids=input_ids, attention_mask=mask)
            preds = torch.argmax(out.logits, dim=1).cpu().numpy()
            v_preds.extend(preds)
            v_trues.extend(labels.cpu().numpy())

    vf1 = f1_score(v_trues, v_preds, pos_label=1, zero_division=0)
    vp = precision_score(v_trues, v_preds, pos_label=1, zero_division=0)
    vr = recall_score(v_trues, v_preds, pos_label=1, zero_division=0)
    elapsed = time.time() - t0

    print(f'Epoch {epoch+1}/{EPOCHS} [{elapsed:.1f}s] | Train Loss: {avg_loss:.4f} | Val F1: {vf1:.4f} (P={vp*100:.1f}%, R={vr*100:.1f}%)')
    if vf1 > best_val_f1:
        best_val_f1 = vf1
        torch.save(model.state_dict(), best_ckpt)
        print(f' -> 🌟 Đã lưu checkpoint mới tốt nhất: Val F1 = {best_val_f1:.4f}')

print('\n✅ Đã nạp checkpoint tốt nhất để đánh giá kiểm thử!')
model.load_state_dict(torch.load(best_ckpt))

In [ ]:
# BƯỚC 6: HIỆU CHUẨN NGƯỠNG TRÊN VAL VÀ ĐÁNH GIÁ TRÊN TEST (2.000 MẪU)
from sklearn.metrics import average_precision_score, confusion_matrix

model.eval()
val_probs = []
with torch.no_grad():
    for batch in val_loader:
        input_ids = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        out = model(input_ids=input_ids, attention_mask=mask)
        p = torch.softmax(out.logits, dim=1)[:, 1].cpu().numpy()
        val_probs.extend(p)

val_probs = np.array(val_probs)
val_trues = df_val['label'].values

# Quét ngưỡng tối ưu trên tập Validation
best_th, best_vf1 = 0.5, -1
for th in np.arange(0.1, 0.9, 0.02):
    f1 = f1_score(val_trues, (val_probs >= th).astype(int), pos_label=1, zero_division=0)
    if f1 > best_vf1:
        best_vf1, best_th = f1, th

print(f'🎯 Ngưỡng tối ưu xác định trên Val: Threshold = {best_th:.2f} (F1 Val = {best_vf1:.4f})')

# Dự đoán trên tập Test
test_probs = []
with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        out = model(input_ids=input_ids, attention_mask=mask)
        p = torch.softmax(out.logits, dim=1)[:, 1].cpu().numpy()
        test_probs.extend(p)

test_probs = np.array(test_probs)
test_preds = (test_probs >= best_th).astype(int)
y_test = df_test['label'].values

acc = (test_preds == y_test).mean()
prec = precision_score(y_test, test_preds, pos_label=1, zero_division=0)
rec = recall_score(y_test, test_preds, pos_label=1, zero_division=0)
f1 = f1_score(y_test, test_preds, pos_label=1, zero_division=0)
macro_f1 = f1_score(y_test, test_preds, average='macro', zero_division=0)
prauc = average_precision_score(y_test, test_probs)
tn, fp, fn, tp = confusion_matrix(y_test, test_preds).ravel()

print('\n' + '=' * 75)
print('     KẾT QUẢ PhoBERT FINE-TUNED TRÊN TẬP TEST ĐIỆN MÁY XANH (2.000 MẪU)')
print('=' * 75)
print(f'Accuracy         : {acc*100:.2f}%')
print(f'Precision (Toxic): {prec*100:.2f}%')
print(f'Recall (Toxic)   : {rec*100:.2f}% (Bắt đúng {tp}/75 câu)')
print(f'F1-Score (Toxic) : {f1:.4f}')
print(f'Macro F1-Score   : {macro_f1:.4f}')
print(f'PR-AUC           : {prauc:.4f}')
print(f'Ma trận nhầm lẫn : TP = {tp}, FP = {fp}, FN = {fn}, TN = {tn}')

In [ ]:
# BƯỚC 7: XUẤT FILE PHÙ HỢP QUY ƯỚC NỘP TV4 VÀ TỰ ĐỘNG TẢI VỀ
import json
from google.colab import files

# 1. Xuất phobert.csv
df_out = pd.DataFrame({
    'review_id': df_test['review_id'],
    'label': y_test,
    'score': np.round(test_probs, 4),
    'pred': test_preds
})
df_out.to_csv('phobert.csv', index=False, encoding='utf-8-sig')

# 2. Xuất phobert_info.json
info_data = {
    'nguoi_phu_trach': 'TV3',
    'threshold': float(best_th),
    'chon_nguong_tren': 'tap_validation',
    'sieu_tham_so': {
        'backbone': 'vinai/phobert-base-v2',
        'training_mode': 'End-to-End Fine-Tuning (3 epochs, lr=2e-5, AdamW, GPU T4)',
        'loss': 'CrossEntropyLoss(weights=[1.0, 8.0])',
        'max_len': 64,
        'batch_size': 32
    },
    'random_seed': 42,
    'metrics_test': {
        'accuracy': float(acc),
        'precision_toxic': float(prec),
        'recall_toxic': float(rec),
        'f1_toxic': float(f1),
        'macro_f1': float(macro_f1),
        'pr_auc': float(prauc),
        'tp': int(tp), 'fp': int(fp), 'fn': int(fn), 'tn': int(tn)
    }
}

with open('phobert_info.json', 'w', encoding='utf-8') as f:
    json.dump(info_data, f, indent=2, ensure_ascii=False)

print('\n📥 Đang kích hoạt tự động tải 2 file về máy của bạn...')
files.download('phobert.csv')
files.download('phobert_info.json')
print('✅ XONG! Sau khi tải về, bạn chỉ cần ném 2 file này vào thư mục results/predictions/ trên máy nhé!')